In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np

from util import Pipeline, load_base_year_emp
from util.elmer_helpers import read_from_elmer_geo, read_from_elmer

p = Pipeline('configs')

In [2]:
county = p.get_geodataframe('county').query("psrc == 1")

reg = p.get_geodataframe('regional_geographies').clip(county.dissolve())
reg = reg.reset_index(drop=True)
reg['reg_id'] = reg.index + 1
reg['reg'] = 1

included_milspn_ids = [12,13,19,20,21,22]
military = p.get_geodataframe('military_bases').query("milspn_id in @included_milspn_ids").dissolve('milspn_id').reset_index(drop=True)
military['military_id'] = military.index + 1
military = military.clip(county.dissolve())
military = military.rename(columns={'name':'military_name'})

tribal = p.get_geodataframe('tribal_land').clip(county.dissolve())
tribal = tribal.loc[tribal.tribal_land=='Tulalip Reservation'].dissolve()
tribal['tribal_id'] = tribal.index + 1

nat_resource = p.get_geodataframe('natural_resource').clip(county.dissolve()).dissolve('resource',as_index=False).reset_index(drop=True)
nat_resource['nr_id'] = nat_resource.index + 1


In [3]:
def union_dissolve(primary_gdf, secondary_gdf, primary_id_col, secondary_id_col, name1, name2):
    flags = ['reg','military','tribal','nr','jblm']
    flag_cols_1 = [col for col in flags if col in primary_gdf.columns]
    print(flag_cols_1)
    gdf1 = primary_gdf[[primary_id_col,name1, 'geometry'] + flag_cols_1].copy()
    gdf1[primary_id_col] = gdf1[primary_id_col].astype(str) + '_' + primary_id_col.split('_id')[0]
    geog_flag = primary_id_col.split('_id')[0]
    gdf1[geog_flag] = 1
    flag_cols_2 = [col for col in flags if col in secondary_gdf.columns]
    print(flag_cols_2)
    gdf2 = secondary_gdf[[secondary_id_col,name2, 'geometry'] + flag_cols_2].copy()
    gdf2[secondary_id_col] = gdf2[secondary_id_col].astype(str) + '_' + secondary_id_col.split('_id')[0]

    # perform union
    gdf = gpd.overlay(gdf1, gdf2, how='union')

    # for secondary polygons that do not overlap, bring id over to original id column
    # this will allow for bringing in new polygons from secondary layer while keeping original
    # polygons from primary layer intact. Secondary polygons will lose any overlapping area with primary polygons
    # to preserve the original geometry of the primary layer.
    gdf.loc[gdf[primary_id_col].isna(), primary_id_col] = gdf.loc[gdf[primary_id_col].isna(), secondary_id_col]

    # dissolve on primary id column
    gdf = gdf.dissolve(by=primary_id_col, as_index=False)

    combined_layers = primary_id_col.split('_id')[0] + '_' + secondary_id_col.split('_id')[0]
    combined_name = combined_layers + '_name'
    combined_id = combined_layers + '_id'

    gdf[combined_name] = np.where(gdf[name1].notna(), gdf[name1], gdf[name2])
    gdf = gdf.reset_index(drop=True)
    gdf[combined_id] = gdf.index + 1
    return gdf

In [4]:
gdf = union_dissolve(military, reg, 'military_id', 'reg_id','military_name','juris')
gdf.to_file('data/reg_mil.gpkg', driver='GPKG')

[]
['reg']


In [9]:
gdf

,military_id,geometry,military_name,military,reg_id,juris,reg,military_reg_name,military_reg_id
0,100_reg,"POLYGON ((1355035.72 83048.78, 1354885.68 8304...",None,NaN,100_reg,Enumclaw,1.0,Enumclaw,1
1,101_reg,"POLYGON ((1341807.625 127676.047, 1341835.125 ...",None,NaN,101_reg,Black Diamond,1.0,Black Diamond,2
2,102_reg,"MULTIPOLYGON (((1357462.552 121521.88, 1357462...",None,NaN,102_reg,Black Diamond PAA,1.0,Black Diamond PAA,3
3,103_reg,"POLYGON ((1341739.625 145940.438, 1341739.666 ...",None,NaN,103_reg,Maple Valley,1.0,Maple Valley,4
4,104_reg,"POLYGON ((1408363.55 176562.89, 1408360.5 1763...",None,NaN,104_reg,North Bend UGA,1.0,North Bend UGA,5
...,...,...,...,...,...,...,...,...,...
164,96_reg,"POLYGON ((1326305.883 145065.25, 1327597.544 1...",None,NaN,96_reg,Covington PAA,1.0,Covington PAA,165
165,97_reg,"POLYGON ((1347943.259 65869.759, 1347980.215 6...",None,NaN,97_reg,Buckley,1.0,Buckley,166
166,98_reg,"POLYGON ((1355517.625 62115.797, 1355542.75 62...",None,NaN,98_reg,Enumclaw,1.0,Enumclaw,167
167,99_reg,"MULTIPOLYGON (((1346996.61 67792.38, 1346965.5...",None,NaN,99_reg,Enumclaw UGA,1.0,Enumclaw UGA,168


In [8]:
jblm_uga

,juris,geometry,jblm_id
14,JBLM UGA,"POLYGON ((1233671.275 60551.454, 1233675.896 6...",1


In [10]:
jblm_uga = reg.loc[reg['juris'] == 'JBLM UGA'].copy()
jblm_uga = jblm_uga[['juris','geometry']]
jblm_uga['jblm_id'] = 1
gdf = union_dissolve(jblm_uga, gdf, 'jblm_id','military_reg_id','juris','military_reg_name')
gdf.to_file('data/reg_mil_jblm_uga.gpkg', driver='GPKG')

[]
['reg', 'military']


c:\Users\JKolberg\PythonProjects\control_totals\.venv\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 6 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [12]:
jblm = gdf.loc[gdf['jblm_military_reg_name']=='Joint Base Lewis McChord'].copy()
jblm = jblm.dissolve()
gdf = gdf.loc[gdf['jblm_military_reg_name']!='Joint Base Lewis McChord'].copy()
gdf = pd.concat([gdf, jblm], ignore_index=True)
gdf.to_file('data/reg_mil_jblm.gpkg', driver='GPKG')

In [13]:
gdf = union_dissolve(tribal,gdf,'tribal_id','jblm_military_reg_id','tribal_land','jblm_military_reg_name')
gdf.to_file('data/reg_mil_jblm_tribal.gpkg', driver='GPKG')

[]
['reg', 'military', 'jblm']


In [14]:
gdf = union_dissolve(nat_resource, gdf,'nr_id','tribal_jblm_military_reg_id','resource','tribal_jblm_military_reg_name')
gdf.to_file('data/reg_mil_jblm_tribal_nat_resource.gpkg', driver='GPKG')

[]
['reg', 'military', 'tribal', 'jblm']


c:\Users\JKolberg\PythonProjects\control_totals\.venv\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 310 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [15]:
gdf.columns

Index(['nr_id', 'geometry', 'resource', 'nr', 'tribal_jblm_military_reg_id',
       'tribal_jblm_military_reg_name', 'reg', 'military', 'tribal', 'jblm',
       'nr_tribal_jblm_military_reg_name', 'nr_tribal_jblm_military_reg_id'],
      dtype='object')

In [16]:
county['county_id'] = ('53' + county['county_fip']).astype(int)
gdf = union_dissolve(gdf,county,'nr_tribal_jblm_military_reg_id','county_id','nr_tribal_jblm_military_reg_name','county_fip')

['reg', 'military', 'tribal', 'nr', 'jblm']
[]


c:\Users\JKolberg\PythonProjects\control_totals\.venv\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 765 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [17]:
gdf.to_file('data/reg_mil_jblm_tribal_nr_cnty.gpkg', driver='GPKG')

In [ ]:
reg_mil = gpd.overlay(reg, military, how='union')

In [13]:
reg_mil.loc[(reg_mil['juris']=='JBLM UGA') & (reg_mil['name']=='Joint Base Lewis McChord'),'name'] = 'JBLM UGA'

In [38]:
mil_only = reg_mil.loc[~reg_mil.military_id.isna()].copy()
mil_only = mil_only.dissolve('name').reset_index().explode()
mil_only_slivers = mil_only.loc[mil_only.geometry.area < 5000000].copy()
mil_no_slivers = mil_only.loc[mil_only.geometry.area >= 5000000].copy()
mil_only_slivers['sliver_id'] = mil_only_slivers.reset_index(drop=True).index + 1
sliver_pts = mil_only_slivers.copy()
sliver_pts['geometry'] = mil_only_slivers.representative_point()
sliver_pts = sliver_pts[['sliver_id','geometry']].sjoin_nearest(mil_no_slivers[['name','geometry']])
sliver_pts = sliver_pts.rename(columns={'name':'name_nearest'})
sliver_pts.to_file('data/mil_sliver_pts.gpkg', driver='GPKG')
mil_only_slivers = mil_only_slivers[['sliver_id','geometry']].merge(sliver_pts[['sliver_id','name_nearest']], on='sliver_id')
mil = gpd.overlay(mil_only_slivers, mil_no_slivers, how='union')
mil.loc[~mil.name_nearest.isna(),'name'] = mil.loc[~mil.name_nearest.isna(),'name_nearest']
mil = mil.dissolve('name').reset_index()
mil.to_file('data/military_only.gpkg', driver='GPKG')

c:\Users\JKolberg\PythonProjects\control_totals\.venv\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 22 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [140]:
def union_snap_boundary(input_gdf, input_gdf_id, starting_gdf, starting_gdf_id, tolerance):
    input_gdf = gpd.clip(input_gdf, starting_gdf.dissolve())
    m_buffer = input_gdf[[input_gdf_id,'geometry']].copy()
    m_buffer['geometry'] = m_buffer.geometry.buffer(tolerance)

    overlay = gpd.overlay(input_gdf, starting_gdf, how='union')
    overlay = overlay.explode().reset_index(drop=True)
    overlay['exp_id'] = overlay.index

    overlay_pts = overlay.copy()
    overlay_pts['geometry'] = overlay_pts.representative_point()

    overlay_pts = overlay_pts[['exp_id','geometry']].sjoin(m_buffer)
    overlay_pts = overlay_pts.rename(columns={input_gdf_id:f'{input_gdf_id}_buffer'})
    overlay = overlay.merge(overlay_pts[['exp_id',f'{input_gdf_id}_buffer']], on='exp_id',how='left')
    overlay = overlay.dissolve([f'{input_gdf_id}_buffer',starting_gdf_id],dropna=False)
    return overlay

In [132]:
cnty_mil = union_snap_boundary(military, 'military_id',county, 'county_fip', 150)
cnty_mil.to_file('data/cnty_mil.gpkg', driver='GPKG')

In [133]:
cnty_mil = cnty_mil.reset_index()
cnty_mil['cnty_mil_id'] = cnty_mil.index + 1
cnty_mil = cnty_mil[['cnty_mil_id','name','county_fip','geometry']]

In [129]:
cnty_mil.to_file('data/cnty_mil.gpkg', driver='GPKG')

In [142]:
cnty_mil.overlay(reg).to_file('data/cnty_mil_reg.gpkg', driver='GPKG')

In [141]:
cnty_mil_reg = union_snap_boundary(reg, 'reg_id',cnty_mil, 'cnty_mil_id', 150)
cnty_mil_reg.to_file('data/cnty_mil_reg.gpkg', driver='GPKG')

c:\Users\JKolberg\PythonProjects\control_totals\.venv\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 50 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [145]:
reg_mil = union_snap_boundary(military, 'military_id',reg, 'reg_id', 500)
reg_mil.to_file('data/reg_mil.gpkg', driver='GPKG')

c:\Users\JKolberg\PythonProjects\control_totals\.venv\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 26 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)
